# Skull acoustic transparency at **500 kHz** — Colab sketch

Time-reversal skull *transparency maps* + transducer placement, end to end, small enough to run on a free Colab GPU.

**Why 500 kHz fits Colab.** The full-wave grid is set by points-per-wavelength: `dx = c0 / (f0 · ppw)`. At 500 kHz / 6 ppw that is **~0.51 mm**, so a whole human head is **~374×455×373 ≈ 63 M voxels** — about **1/16** the work of the lab's 1 MHz deck (508 M voxels, ~5 min on an A6000). That lands at **~2.5–5 GB of GPU memory and ~1–2 min** on a Colab **T4 (16 GB)**.

**Pipeline (all `skull-transparency` CLI + a GPU solve):**

`prepare` (skull map → `.dat` sim inputs, no GPU) → `sim outward --run` (full-wave GPU solve) → `extract` (→ Field Bundle) → `transparency` (the map) / `place` (a focused window).

> **Status of this sketch.** The **no-GPU analysis path (Section 1)** runs anywhere. The **GPU solve (Section 2)** is laid out with the real commands but uses a **toy synthetic head** and a **CTX-500 spec at 500 kHz**; two things are deliberately flagged as not-yet-final — see *Caveats* at the end (the CTX-500 500 kHz **piston source model** is an open roadmap item, and the toy head should be swapped for a real CT-derived sound-speed map).

## 0 · Runtime

In Colab: **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU** (a free **T4** is plenty at 500 kHz).

In [ ]:
# --- setup: install the Apache-licensed package + fetch the CUDA solver binary ---
import os, sys
IN_COLAB = "google.colab" in sys.modules

# 1) skull_transparency (Apache-2.0): the pure-Python pipeline (numpy/scipy).
!pip -q install "git+https://github.com/pinton-lab/skull_transparency.git"

# 2) The full-wave CUDA solver EXECUTABLE ships in the fullwave2-ultra repo
#    (PolyForm Noncommercial -- academic/non-commercial use). Clone for bin/bench_3d_opt.
!git clone --depth 1 https://github.com/pinton-lab/fullwave2-ultra.git /content/fullwave2-ultra 2>/dev/null || true
os.environ["FULLWAVE2_BIN"] = "/content/fullwave2-ultra/bin/bench_3d_opt"
print("solver binary:", os.environ["FULLWAVE2_BIN"],
      "(exists:", os.path.exists(os.environ["FULLWAVE2_BIN"]), ")")
# (Optional, only for real MNI<->subject registration: pip install git+.../tuba.git)

In [ ]:
# --- confirm a GPU is attached (needed for Section 2 only) ---
!nvidia-smi -L || echo "NO GPU -- Section 1 still runs; enable a GPU for Section 2."

## 1 · Quick taste — no GPU, no data

Everything downstream of the solve (transparency map + placement + figures) on a shipped **synthetic Field Bundle**. This cell runs anywhere `skull-transparency` is installed.

In [ ]:
import matplotlib.pyplot as plt
import skull_transparency as st

# a tiny in-package synthetic Field Bundle (stands in for a solved run)
bundle = st.load_bundle(st.make_synthetic_bundle("synthetic_bundle"))

# transparency map: distance-corrected transmitted amplitude per skull-surface patch
tmap = st.compute_transparency_map(bundle)
print(f"{len(tmap.surf_vox)} surface patches")

# place a focused bowl subject to an incidence cap
pl = st.place_bowl(tmap, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=15.0,
                                            theta_max_deg=35.0))
score = st.PositioningScore.from_placement(pl, target_name="synthetic_target")
print(f"placement incidence {pl.incidence_deg:.1f} deg, score {score.normalized:.3f}")

# render the transparency surface inline (dB log-|p|)
st.render_transparency_surface(tmap, "transparency_synth.png",
                               title="Synthetic skull transparency (no-GPU demo)")
plt.figure(figsize=(7, 5)); plt.imshow(plt.imread("transparency_synth.png")); plt.axis("off"); plt.show()

## 2 · Full 500 kHz simulation — `prepare → solve → extract`

Real full-wave run on a GPU. At 500 kHz the grid is coarse enough (~0.5 mm, ~63 M voxels) to fit a T4 and finish in ~1–2 min.

### 2a · A sound-speed map to solve on

**Toy placeholder** — a water ball with a bone shell — so the chain runs without any dataset. **For real work, replace this with a CT-derived sound-speed map** (`--c-map your_c.nii.gz` carrying its own voxel→world affine; e.g. the public Halle micro-CT via `tuba`).

In [ ]:
import numpy as np
# ~120 mm cube at 0.5 mm -> 240^3; water c=1500, ~6 mm bone shell at c=2600 m/s
n, vox_mm = 240, 0.5
c = np.full((n, n, n), 1500.0, np.float32)
zz, yy, xx = np.mgrid[0:n, 0:n, 0:n]
r = np.sqrt((xx - n/2)**2 + (yy - n/2)**2 + (zz - n/2)**2) * vox_mm
c[(r > 48) & (r < 54)] = 2600.0                      # bone shell (skull stand-in)
np.save("head_c.npy", c)
affine = np.diag([vox_mm, vox_mm, vox_mm, 1.0]).astype(np.float64)  # voxel->world mm
affine[:3, 3] = -vox_mm * n / 2                       # center the world origin
np.save("affine.npy", affine)
print("toy c-map:", c.shape, "unique c:", np.unique(c))

### 2b · `prepare` — skull map → solver inputs (no GPU)

Frequency lives in the **TransducerSpec**, passed inline. `{"preset":"ctx500","f0_hz":500000}` builds the CTX-500 geometry at **500 kHz** (`dx = 1540/(500e3·6) ≈ 0.51 mm`). `--center` seats an omnidirectional source at the brain center for the neutral **whole-skull** transparency map (drop it and pass `--target x,y,z --approach x,y,z` to aim at a specific target).

In [ ]:
!skull-transparency prepare \
    --c-map head_c.npy --affine affine.npy --center \
    --transducer '{"preset":"ctx500","f0_hz":500000}' \
    --out run
!echo "--- sim tree ---" && ls run && echo "grid meta:" && cat run/meta.json 2>/dev/null | head -20

### 2c · Outward full-wave solve (GPU)

Streams the outward field to the recorders + a decimated full-field dump (`genout_mod.dat`, which `extract` reads). ~1–2 min on a T4 at 500 kHz.

In [ ]:
!python -m skull_transparency.sim outward --sim run --out run --run
!ls -lh run/outward/genout_mod.dat 2>/dev/null || echo "no genout_mod.dat -- did the solve run on a GPU?"

### 2d · Extract → Field Bundle → transparency map

In [ ]:
!skull-transparency extract --run run/outward --sim run --out run/bundle

import matplotlib.pyplot as plt
import skull_transparency as st
b = st.load_bundle("run/bundle")
tmap = st.compute_transparency_map(b)
st.render_transparency_surface(tmap, "transparency_500kHz.png",
                               title="Skull transparency @ 500 kHz (whole-skull, brain-center source)")
plt.figure(figsize=(8, 6)); plt.imshow(plt.imread("transparency_500kHz.png")); plt.axis("off"); plt.show()
print(f"{len(tmap.surf_vox)} surface patches; brighter = more transparent bone")

### 2e · Place a focused window on the map

In [ ]:
pl = st.place_bowl(tmap, st.BowlConstraints(focal_length_mm=60.0, bowl_radius_mm=32.0,
                                            theta_max_deg=35.0))
score = st.PositioningScore.from_placement(pl, target_name="brain_center")
print(f"incidence {pl.incidence_deg:.1f} deg, score {score.normalized:.3f}")

## Caveats & next steps

- **CTX-500 500 kHz source model is an open item.** This uses the CTX-500 *spec* (`preset:ctx500`), but the faithful **4-annular piston** source at 500 kHz (surface-vs-piston) is still on the roadmap. For a first pass, a generic bowl source at 500 kHz (`{"f0_hz":500000,"geometry":"bowl","roc_mm":...,"aperture_mm":...}`) is the safe choice; treat the annular fidelity as pending validation.
- **The toy head is a placeholder.** Swap Section 2a for a real CT-derived sound-speed map (`--c-map c.nii.gz` with its affine). The public **Halle** micro-CT (Zenodo) is fetchable via `tuba`; a real map also lets you set `--bone-threshold` and aim at named MNI targets.
- **Binary license.** `bench_3d_opt` is **PolyForm Noncommercial** — fine for academic/Colab use; commercial use needs a separate license. The Python package is Apache-2.0.
- **Disk/session.** At 500 kHz `genout_mod.dat` is small (coarse grid), so Colab's disk and session limits are not a concern (unlike the 1 MHz deck). Free-tier GPU availability is variable but a <2 min solve is easy.
- **Accuracy at low ppw.** If you want it even faster, the re-optimized `m6` / `M8+2rh` stencils hold dispersion at lower ppw — but validate them on a skull before trusting numbers (they're physics-gated, not skull-gated yet).